# FraudGraph - Phase 4: Operational Fraud Explainer

**Goal:** Every decision comes with actionable, plain-text reasons that fraud operations analysts can understand.
- **Leakage Prevention:** Entity statistics (card, merchant, device) are fitted strictly on the train partition.
- **Lift Thresholds:** High transaction volume without statistical lift is never marked as a risk signal.
- **Reason Types:**
  - Velocity burst (rapid multi-card transactions within 5m)
  - Device shared across cards with prior confirmed fraud
  - Merchant location with elevated chargeback lift
  - Amount anomalies relative to card history
  - Protective reasons (established trusted transaction history)
- **Speed Budget:** Decision explanation in < 1 ms.


In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

if os.path.exists('/kaggle/input/ieee-fraud-detection'):
    DATA_DIR = Path('/kaggle/input/ieee-fraud-detection')
    ARTIFACTS_DIR = Path('/kaggle/working/artifacts')
else:
    DATA_DIR = Path('../data')
    ARTIFACTS_DIR = Path('../artifacts')

from src.explainer import FraudExplainer, build_explainer_stats


## 1. Load Train Split and Fit Explainer Statistics
Strict rule: compute statistics ONLY from the training ledger.

In [ ]:
with open(ARTIFACTS_DIR / 'split_boundaries.json') as f:
    split_bounds = json.load(f)

df_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv')
id_file = DATA_DIR / 'train_identity.csv'
if id_file.exists():
    df_id = pd.read_csv(id_file)
    df = df_tx.merge(df_id, on='TransactionID', how='left')
else:
    df = df_tx

df = df.sort_values(by='TransactionDT').reset_index(drop=True)
df_train = df.iloc[:split_bounds['n_train']]
df_test = df.iloc[split_bounds['n_train'] + split_bounds['n_val']:]

stats = build_explainer_stats(df_train, artifacts_dir=str(ARTIFACTS_DIR))
explainer = FraudExplainer(stats)
print(f'Base fraud rate: {stats["base_fraud_rate"]:.4%}')


## 2. Benchmark Explainer Latency (< 1 ms Budget)

In [ ]:
sample_payload = df_test.iloc[0].to_dict()
vel_info = {'tx_count_5m': 3, 'prior_fraud_count': 1}

# Warmup
explainer.explain(sample_payload, vel_info)

# Measure 1,000 calls
start = time.perf_counter()
for _ in range(1000):
    reasons = explainer.explain(sample_payload, vel_info)
elapsed_ms = (time.perf_counter() - start) * 1000 / 1000

print(f'Average explainer latency: {elapsed_ms:.3f} ms (Target: < 1.0 ms)')
assert elapsed_ms < 1.0, f'Explainer exceeds 1ms target: {elapsed_ms} ms'
print('Sample output reasons:', json.dumps(reasons, indent=2))


## 3. Faithfulness Check on Confirmed Fraud Cases
Evaluate percentage of confirmed fraud payments that receive a concrete relational or velocity risk reason (Target: ~80%).

In [ ]:
fraud_test_sample = df_test[df_test['isFraud'] == 1].head(200)
has_actionable_reason = 0

for _, row in fraud_test_sample.iterrows():
    res = explainer.explain(row.to_dict())
    factors = [r['factor'] for r in res]
    if any(f in factors for f in ['velocity_burst', 'high_risk_card_history', 'suspicious_shared_device', 'high_risk_device', 'high_risk_merchant_cluster', 'abnormal_amount_for_card']):
        has_actionable_reason += 1

faithfulness_ratio = has_actionable_reason / len(fraud_test_sample)
print(f'Actionable reason coverage on confirmed fraud: {faithfulness_ratio:.2%} (~80% target)')
